In [ ]:
import numpy as np
import pandas as pd
import altair as alt
import scipy.sparse as sp


def plot_moments(means_real, means_sim, sd_real, sd_sim, log_scale=True):
    """Compare real vs. simulated means and standard deviations gene-wise.

    Parameters
    ----------
    log_scale : bool
        If True, log-transform all values before plotting (useful when
        counts span several orders of magnitude).
    """
    if log_scale:
        means_real, means_sim = np.log(means_real), np.log(means_sim)
        sd_real, sd_sim = np.log(sd_real), np.log(sd_sim)
    df = pd.DataFrame({
        "means_real": means_real,
        "means_sim": means_sim,
        "sd_real": sd_real,
        "sd_sim": sd_sim,
    })

    means_range = [min(df["means_real"].min(), df["means_sim"].min()),
                   max(df["means_real"].max(), df["means_sim"].max())]
    sd_range = [min(df["sd_real"].min(), df["sd_sim"].min()),
                max(df["sd_real"].max(), df["sd_sim"].max())]

    def identity_line(domain):
        return alt.Chart(pd.DataFrame({"v": domain})).mark_line(
            color="black", strokeDash=[4, 4], opacity=0.5
        ).encode(x="v:Q", y="v:Q")

    means_chart = (
        alt.Chart(df).mark_circle(size=50).encode(
            x=alt.X("means_real", title="Real Means"),
            y=alt.Y("means_sim", title="Simulated Means"),
        ) + identity_line(means_range)
    ).properties(title="Means: Real vs. Simulated")

    sd_chart = (
        alt.Chart(df).mark_circle(size=50).encode(
            x=alt.X("sd_real", title="Real Variances"),
            y=alt.Y("sd_sim", title="Simulated Variances"),
        ) + identity_line(sd_range)
    ).properties(title="Variances: Real vs. Simulated")

    return means_chart | sd_chart

## Interactions Basis

In [ ]:
import anndata
from scipy.stats import zscore

example_sce = anndata.read_h5ad("data/ACINAR_sce.h5ad")
example_sce.obs["spatial1"] = zscore(example_sce.obs["spatial1"])
example_sce.obs["spatial2"] = zscore(example_sce.obs["spatial2"])

In [ ]:
from scdesigner.data import add_spatial_basis, basis_formula
from scdesigner.simulators import PenalizedNegBinCopula

example_sce, mean_cols, mean_pen_diag = add_spatial_basis(
    example_sce, method="tps", df=400, prefix="sp_mean_",
    standardize=True,
)
example_sce, disp_cols, disp_pen_diag = add_spatial_basis(
    example_sce, method="tps", df=100, prefix="sp_disp_",
    standardize=True,
)

# Compute per-gene means for adaptive penalty
n_cell_types = example_sce.obs["cell_type"].nunique()
X_real = example_sce.X
gene_means = X_real.mean(axis=0).flatten()

# use the means in the penalized NB
sim_pen = PenalizedNegBinCopula(
    mean_formula=basis_formula(mean_cols, extra_terms=["cell_type"]),
    dispersion_formula=basis_formula(disp_cols),
    copula_formula="~ 1",
    mean_penalty_diag=mean_pen_diag,
    disp_penalty_diag=disp_pen_diag,
    n_parametric=n_cell_types,
    n_parametric_disp=1, # intercept term
    lam=0.01,
    lam_disp=0.1,
    gene_means=gene_means
)

sim_pen.fit(example_sce, max_epochs=200)

In [ ]:
means_real = X_real.mean(axis=0).flatten()
sd_real = X_real.std(axis=0).flatten()

samples = sim_pen.sample()
means_sim = samples.X.mean(axis=0)
sd_sim = samples.X.std(axis=0)

plot_moments(means_real, means_sim, sd_real, sd_sim, log_scale=True)

In [ ]:
from scdesigner.simulators import NegBinCopula

# train simulator
sim_orig = NegBinCopula(
    mean_formula="~ 0 + cell_type + bs(spatial1, df=5, include_intercept=False) * bs(spatial2, df=5, include_intercept=False)",
    dispersion_formula="~ 0 + bs(spatial1, df=3, include_intercept=False) * bs(spatial2, df=3, include_intercept=False)",
    copula_formula="~ 1"
)
sim_orig.fit(example_sce, max_epochs=200)

In [ ]:
# extract samples
samples_orig = sim_orig.sample()
means_sim = samples.X.mean(axis=0)
sd_sim = samples.X.std(axis=0)

# visualize
plot_moments(means_real, means_sim, sd_real, sd_sim, log_scale=True)

## Visualizations

In [ ]:
# Find genes with largest variance in both datasets
X_real = example_sce.X.toarray() if sp.issparse(example_sce.X) else np.array(example_sce.X)
X_sim = samples.X.toarray() if sp.issparse(samples.X) else np.array(samples.X)

var_real = np.var(X_real, axis=0)
var_sim = np.var(X_sim, axis=0)

# Create scatterplots of real vs. sim data
for gene_idx in [0, 26, 738]:
    for name, idx, X in [("Real Data", gene_idx, X_real), ("Simulated Data", gene_idx, X_sim)]:
        adata = example_sce if name == "Real Data" else samples
        plot_data = pd.DataFrame({
            "spatial1": adata.obs["spatial1"].values,
            "spatial2": adata.obs["spatial2"].values,
            "expr": np.log(X[:, idx] + 1)
        })

        chart = alt.Chart(plot_data).mark_circle(size=30).encode(
            x=alt.X("spatial1", title="Spatial Coordinate 1"),
            y=alt.Y("spatial2", title="Spatial Coordinate 2"),
            color=alt.Color("expr:Q", title="log(Expression + 1)")
        ).properties(
            width=400,
            height=400,
            title=f"{name} - Gene {idx}"
        )

        display(chart)

## Mean Surface Visualization

In [ ]:
from scipy.interpolate import RBFInterpolator


def make_grid_obs(adata, basis_cols, n_grid=30):
    s1 = adata.obs["spatial1"].values
    s2 = adata.obs["spatial2"].values
    g1 = np.linspace(s1.min(), s1.max(), n_grid)
    g2 = np.linspace(s2.min(), s2.max(), n_grid)
    G1, G2 = np.meshgrid(g1, g2, indexing="ij")

    grid_base = pd.DataFrame({
        "spatial1": G1.ravel(),
        "spatial2": G2.ravel(),
        "bin1": pd.cut(G1.ravel(), bins=n_grid, labels=False, include_lowest=True),
        "bin2": pd.cut(G2.ravel(), bins=n_grid, labels=False, include_lowest=True),
    })
    obs_bins = (
        pd.DataFrame({
            "bin1": pd.cut(s1, bins=n_grid, labels=False, include_lowest=True),
            "bin2": pd.cut(s2, bins=n_grid, labels=False, include_lowest=True),
        })
        .dropna()
        .astype(int)
        .drop_duplicates()
    )
    occupied = set(map(tuple, obs_bins[["bin1", "bin2"]].to_numpy()))
    keep = pd.Series(list(zip(grid_base["bin1"], grid_base["bin2"]))).isin(occupied)
    grid_base = grid_base.loc[keep, ["spatial1", "spatial2"]].reset_index(drop=True)

    cell_types = list(adata.obs["cell_type"].unique())
    n_pts = len(grid_base)
    grid_obs = pd.DataFrame({
        "spatial1": np.tile(grid_base["spatial1"].values, len(cell_types)),
        "spatial2": np.tile(grid_base["spatial2"].values, len(cell_types)),
        "cell_type": np.repeat(cell_types, n_pts),
    })

    train_coords = adata.obs[["spatial1", "spatial2"]].values
    grid_coords = grid_obs[["spatial1", "spatial2"]].values
    interp = RBFInterpolator(train_coords, adata.obs[basis_cols].values, neighbors=50)
    grid_basis = interp(grid_coords)
    for i, col in enumerate(basis_cols):
        grid_obs[col] = grid_basis[:, i]
    return grid_obs


def dense_gene_counts(adata, gene_idx):
    if sp.issparse(adata.X):
        return adata.X[:, gene_idx].toarray().ravel()
    return np.array(adata.X)[:, gene_idx]


def binned_obs_df(adata, gene_idx, n_obs_bins, value_col):
    s1 = adata.obs["spatial1"].values
    s2 = adata.obs["spatial2"].values
    return pd.DataFrame({
        "spatial1": pd.cut(s1, bins=n_obs_bins).map(lambda b: round(b.mid, 2)).astype(float),
        "spatial2": pd.cut(s2, bins=n_obs_bins).map(lambda b: round(b.mid, 2)).astype(float),
        value_col: dense_gene_counts(adata, gene_idx),
    })


def fitted_surface_df(sim, adata, basis_cols, gene_idx, pred_key, value_col, n_grid):
    grid_obs = make_grid_obs(adata, basis_cols, n_grid)
    n_ct = adata.obs["cell_type"].nunique()
    n_pts = len(grid_obs) // n_ct
    pred = np.log1p(sim.predict(obs=grid_obs)[pred_key][:, gene_idx])
    pred_avg = pred.reshape(n_ct, n_pts).mean(axis=0)
    ref = grid_obs.iloc[:n_pts]
    return pd.DataFrame({
        "spatial1": ref["spatial1"].round(2).values,
        "spatial2": ref["spatial2"].round(2).values,
        value_col: pred_avg,
    })


def surface_heatmaps(fitted_df, obs_df, value_col, color_title, fitted_title, obs_title):
    vmax = max(fitted_df[value_col].max(), obs_df[value_col].max())
    scale = alt.Scale(domain=[0, vmax], scheme="viridis")

    def heatmap(df, title):
        return (
            alt.Chart(df).mark_rect().encode(
                x=alt.X("spatial1:O", title="spatial1"),
                y=alt.Y("spatial2:O", title="spatial2"),
                color=alt.Color(f"{value_col}:Q", scale=scale, title=color_title),
            ).properties(width=300, height=300, title=title)
        )

    return heatmap(fitted_df, fitted_title) | heatmap(obs_df, obs_title)


def plot_mean_surface(sim, adata, basis_cols, gene_idx, n_grid=30, n_obs_bins=30):
    fitted_df = fitted_surface_df(
        sim, adata, basis_cols, gene_idx,
        pred_key="mean", value_col="mu", n_grid=n_grid,
    )
    obs_df = (
        binned_obs_df(adata, gene_idx, n_obs_bins, value_col="mu")
        .groupby(["spatial1", "spatial2"], as_index=False)["mu"].mean()
        .assign(mu=lambda d: np.log1p(d["mu"]))
    )
    plots = surface_heatmaps(
        fitted_df, obs_df, value_col="mu", color_title="log(mu+1)",
        fitted_title=f"Fitted mean(x) - Gene {gene_idx}",
        obs_title=f"Observed bin mean - Gene {gene_idx}",
    )
    return fitted_df, plots


def plot_dispersion_surface(sim, adata, basis_cols, gene_idx, n_grid=10, n_obs_bins=10):
    fitted_df = fitted_surface_df(
        sim, adata, basis_cols, gene_idx,
        pred_key="dispersion", value_col="dispersion", n_grid=n_grid,
    )
    obs_df = (
        binned_obs_df(adata, gene_idx, n_obs_bins, value_col="count")
        .groupby(["spatial1", "spatial2"], as_index=False)["count"]
        .agg(mu="mean", var="var")
        .assign(dispersion=lambda d: np.where(d["var"] > d["mu"], d["mu"] ** 2 / (d["var"] - d["mu"]), np.nan))
        .assign(dispersion=lambda d: np.log1p(d["dispersion"]))
        .dropna(subset=["dispersion"])
    )
    plots = surface_heatmaps(
        fitted_df, obs_df, value_col="dispersion", color_title="log(dispersion+1)",
        fitted_title=f"Fitted dispersion(x) - Gene {gene_idx}",
        obs_title=f"Observed bin dispersion - Gene {gene_idx}",
    )
    return fitted_df, plots

In [ ]:
for gene_idx in [0, 26, 738]:
    df, plots = plot_mean_surface(sim_pen, example_sce, mean_cols + disp_cols, gene_idx)
    disp_df, disp_plots = plot_dispersion_surface(sim_pen, example_sce, mean_cols + disp_cols, gene_idx)
    display(plots)
    display(disp_plots)